# Reto 1 — Hallazgos: Estudiantes 2025 vs 2026

**Objetivo.** Describir (qué pasó) y diagnosticar (por qué pudo pasar) la variación
en el uso de CREA entre 2025 y 2026 en la población de estudiantes.

**Cómo leer este notebook.** Cada sección tiene un texto que explica *qué muestra* la
figura y *cómo interpretarla*, y una celda de código que calcula la lectura con los
datos reales e imprime un resumen debajo del gráfico. Las conclusiones finales las
escribís vos a partir de esos números: el notebook te da el marco, no el veredicto.

## Nota metodológica (leer antes de sacar conclusiones)

- **Matrícula ≠ acceso.** Cada fila es un estudiante *matriculado*; que esté en la
  tabla no significa que haya entrado a CREA. Medimos acceso con `accedio`
  (`dias_totales > 0`). Al leer una caída, siempre aclarar si es de matrícula o de uso.
- **Ahora hay panel longitudinal.** Los IDs de estudiante son únicos y estables entre
  años, así que seguimos a la *misma persona* 2025→2026. Eso habilita medir retención y
  abandono, no solo comparar cohortes.
- **No se cruza con docentes por centro.** La anonimización rompió el `ID_CENTRO` entre
  tablas; los cortes por centro valen *dentro* de estudiantes, no para emparejar con docentes.
- **Vulnerabilidad por nivel.** `vuln_q` (1–5, 1 = más vulnerable) combina `CONTEXTO`
  (primaria) e `IVSMEDIA` (media), que son disjuntos por nivel. Puede haber nulos.
- **Privacidad.** Los cortes por grupo suprimen los grupos con menos de `MIN_N`
  personas.

In [ ]:
import pathsetup  # raíz del repo en sys.path (notebooks en reportes/)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# >>> AJUSTAR si hace falta: carpeta con los parquet del pipeline <<<
CARPETA = Path(r"C:\Users\Matihas\Desktop\Datos Ceibal 2025-2026\processed")
MIN_N   = 10                     # umbral minimo por grupo (privacidad)
C25, C26 = "#4C72B0", "#DD8452"  # colores 2025 / 2026

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": .3, "axes.axisbelow": True})

e25   = pd.read_parquet(CARPETA / "estudiantes_2025_clean.parquet")
e26   = pd.read_parquet(CARPETA / "estudiantes_2026_clean.parquet")
panel = pd.read_parquet(CARPETA / "panel_estudiantes.parquet")

print("2025 :", e25.shape)
print("2026 :", e26.shape)
print("panel:", panel.shape)
e25.head(3)


## 1. Panorama: matrícula vs acceso

La primera pregunta es *qué* cayó: ¿hay menos estudiantes en el sistema, o los mismos
entran menos a CREA? Esta figura separa las dos cosas. Si la matrícula baja pero la
proporción que accede se mantiene, buena parte del descenso es cambio poblacional
(egresos, altas/bajas) más que desenganche de la plataforma. Si además cae el acceso,
ahí sí hay una señal de uso.

In [ ]:
n25, n26 = len(e25), len(e26)
a25, a26 = int(e25["accedio"].sum()), int(e26["accedio"].sum())

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(2); w = .38
ax.bar(x - w/2, [n25, a25], w, label="2025", color=C25)
ax.bar(x + w/2, [n26, a26], w, label="2026", color=C26)
ax.set_xticks(x); ax.set_xticklabels(["Matriculados", "Accedieron"])
ax.set_ylabel("Estudiantes"); ax.legend(); ax.set_title("Matrícula vs acceso")
for i, (u, v) in enumerate(zip([n25, a25], [n26, a26])):
    ax.text(i - w/2, u, f"{u:,}", ha="center", va="bottom", fontsize=8)
    ax.text(i + w/2, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
plt.show()

print(f"Matrícula: {n25:,} -> {n26:,}  ({(n26/n25-1)*100:+.1f}%)")
print(f"Accedieron: {a25:,} -> {a26:,}  ({(a26/a25-1)*100:+.1f}%)")
print("Lectura -> comparar ambas variaciones: si la de acceso es MÁS negativa que la de "
      "matrícula, hay desenganche; si son parecidas, domina el cambio poblacional.")

## 2. Tasa de acceso

La tasa (% que accedió al menos un día) controla por el tamaño de la población: es la
medida más limpia de *uso*. Una caída de la tasa indica que, entre los que están, una
proporción mayor no entró a CREA — eso no lo explica el egreso.

In [ ]:
t25, t26 = 100*e25["accedio"].mean(), 100*e26["accedio"].mean()
fig, ax = plt.subplots(figsize=(4.2, 4))
ax.bar(["2025", "2026"], [t25, t26], color=[C25, C26])
for i, v in enumerate([t25, t26]):
    ax.text(i, v, f"{v:.1f}%", ha="center", va="bottom")
ax.set_ylabel("% que accedió ≥1 día"); ax.set_title("Tasa de acceso")
plt.show()
print(f"Tasa de acceso: {t25:.1f}% (2025) -> {t26:.1f}% (2026), cambio {t26-t25:+.1f} pp")
print("Lectura -> si baja, es señal de uso real, independiente del tamaño de la cohorte.")

## 3. Distribución de días de acceso

Más allá del binario accedió/no, ¿cambió la *intensidad* de uso? Miramos la forma de
`dias_totales` (abril+mayo+junio). Interesa la masa en 0 (no accedieron) y la cola alta
(usuarios intensos). Un corrimiento hacia la izquierda entre años = menos uso por persona.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
top = int(np.nanmax([e25["dias_totales"].max(), e26["dias_totales"].max()]))
bins = np.arange(0, top + 4, 3)
ax.hist(e25["dias_totales"].dropna(), bins=bins, density=True, alpha=.55, label="2025", color=C25)
ax.hist(e26["dias_totales"].dropna(), bins=bins, density=True, alpha=.55, label="2026", color=C26)
ax.set_xlabel("dias_totales"); ax.set_ylabel("densidad"); ax.legend()
ax.set_title("Distribución de días de acceso")
plt.show()
print(f"Mediana días  2025: {e25['dias_totales'].median():.0f} | 2026: {e26['dias_totales'].median():.0f}")
print(f"% en 0 días   2025: {100*(e25['dias_totales']==0).mean():.1f}% | 2026: {100*(e26['dias_totales']==0).mean():.1f}%")
print("Lectura -> comparar medianas y masa en 0; suben los ceros = más gente sin entrar.")

## 4. Panel longitudinal: retenidos, salidas y altas

Como seguimos a la misma persona, clasificamos a cada estudiante en: **ambos**
(está los dos años), **solo 2025** (estaba y ya no) y **solo 2026** (nuevo). Ojo: "solo
2025" mezcla egresos con abandonos del sistema — esta figura sola no los separa; la
sección 6 aísla el desenganche real dentro de los que siguen.

In [ ]:
orden = ["solo_2025", "ambos", "solo_2026"]
vc = panel["estado"].value_counts().reindex(orden).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(5.2, 4))
ax.bar(["Solo 2025\n(salió)", "Ambos\n(retenido)", "Solo 2026\n(nuevo)"], vc.values,
       color=["#C44E52", "#55A868", "#8172B3"])
for i, v in enumerate(vc.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("Estudiantes"); ax.set_title("Composición del panel")
plt.show()
print(vc.to_string())
print("Lectura -> 'solo_2025' = egreso + abandono (no separable acá); 'ambos' habilita medir retención.")

## 5. Acceso por vulnerabilidad (equidad)

Pregunta diagnóstica clave: ¿la caída golpeó parejo, o más a los contextos vulnerables?
`vuln_q` va de 1 (más vulnerable) a 5. Miramos la tasa de acceso por quintil en cada
año; un deterioro concentrado en los quintiles bajos es un hallazgo de equidad relevante.
(Se suprimen quintiles con menos de `MIN_N` estudiantes.)

In [ ]:
def tasa_por(col):
    a = e25.groupby(col)["accedio"].agg(["mean", "size"])
    b = e26.groupby(col)["accedio"].agg(["mean", "size"])
    a, b = a[a["size"] >= MIN_N], b[b["size"] >= MIN_N]
    idx = sorted(set(a.index) & set(b.index), key=lambda z: str(z))
    return idx, a, b

idx, a, b = tasa_por("vuln_q")
x = np.arange(len(idx)); w = .38
fig, ax = plt.subplots(figsize=(max(5, len(idx)*1.1), 4))
ax.bar(x - w/2, [100*a.loc[i, "mean"] for i in idx], w, label="2025", color=C25)
ax.bar(x + w/2, [100*b.loc[i, "mean"] for i in idx], w, label="2026", color=C26)
ax.set_xticks(x); ax.set_xticklabels([str(i) for i in idx])
ax.set_xlabel("quintil de vulnerabilidad (1 = más vulnerable)")
ax.set_ylabel("% acceso"); ax.legend(); ax.set_title("Acceso por vulnerabilidad")
plt.show()
for i in idx:
    print(f"  quintil {i}: {100*a.loc[i,'mean']:.1f}% -> {100*b.loc[i,'mean']:.1f}%  ({100*(b.loc[i,'mean']-a.loc[i,'mean']):+.1f} pp)")
print("Lectura -> comparar la caída entre quintiles; si es mayor en 1-2, hay brecha de equidad.")

## 6. Retención de acceso — la caída "real"

Esta es la figura más importante. Tomamos solo a los **mismos estudiantes** presentes en
ambos años y cruzamos si accedían en 2025 contra si accedieron en 2026. El cuadrante
*accedía 2025 → no accede 2026* es el desenganche genuino: misma persona, seguía en el
sistema, y dejó de entrar. Limpio de egresos y de altas.

In [ ]:
amb = panel[panel["estado"] == "ambos"]
ct = pd.crosstab(amb["accedio_25"], amb["accedio_26"], normalize=True) * 100
fig, ax = plt.subplots(figsize=(4.6, 4))
im = ax.imshow(ct.values, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["no accede 26", "accede 26"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["no accedía 25", "accedía 25"])
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        ax.text(j, i, f"{ct.values[i, j]:.1f}%", ha="center", va="center")
ax.set_title("Retención de acceso (mismos estudiantes)")
fig.colorbar(im, ax=ax, fraction=.046, label="% del total 'ambos'")
plt.show()

base = amb[amb["accedio_25"] == 1]
churn = (base["accedio_26"] == 0).mean() * 100
print(f"De los que accedían en 2025 y siguen en 2026, dejó de acceder: {churn:.1f}%")
print("Lectura -> ese % es el desenganche real; cortalo por vuln_q o Rubro para el diagnóstico.")

## Síntesis y próximos pasos

Para escribir los hallazgos, encadená la lógica de las secciones: (1–2) cuánto de la caída
es población vs uso, (3) si cambió la intensidad, (4) cuánta gente entra y sale,
(5) si hay brecha de equidad, y (6) el desenganche real entre los retenidos.

**Limitaciones a dejar explícitas.** No se puede cruzar con docentes por centro; "solo
2025" no separa egreso de abandono; `vuln_q` combina dos índices por nivel; toda
explicación de la caída es hipótesis, no causa demostrada.

**Docentes (siguiente).** Replicar este notebook usando `docentes_*_persona.parquet` y
`panel_docentes.parquet`. Ahí se suman cortes propios: `n_centros`, `n_materias` y la
materia dictada. Recordá que en docentes los días son por persona (ya colapsados), no por curso.